# Market & AI Pulse: Daily Pipeline Report

A daily health check, regenerated and overwritten fresh every time the pipeline runs. This is not a historical log; it always reflects only the most recent run. For the trailing history of runs and issues, see the git history of this file or the GitHub Actions run list directly.

This notebook answers three questions each day:
1. Did the pipeline actually run, and did it succeed?
2. What data landed, and does it look right for the day (weekends and monthly-cadence sources are expected to be quiet, not broken)?
3. Where does each piece of data actually live, end to end?

Unlike `analysis/key_findings.ipynb`, this notebook queries the Databricks tables directly rather than the published JSON, so it can catch a gap even if the export/publish step itself has a bug. That means running it locally requires the same `DATABRICKS_HOST`/`DATABRICKS_TOKEN` credentials as the rest of the pipeline - it isn't meant to be run casually without them the way the analysis notebook is.

## Setup

In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime, timezone, date, timedelta

import pandas as pd
import requests
from IPython.display import Markdown, display

sys.path.insert(0, str(Path.cwd().parent / 'databricks'))
from warehouse import connect  # noqa: E402

pd.set_option('display.max_colwidth', None)

REPO = 'pdglenchur-glitch/market_ai_pulse'
TODAY = datetime.now(timezone.utc).date()
NOW = datetime.now(timezone.utc)

# Optional: authenticates GitHub API calls when run inside CI (GITHUB_TOKEN is
# already available there), avoiding the much lower unauthenticated rate
# limit. Falls back to unauthenticated for a casual local run.
_gh_token = os.environ.get('GITHUB_TOKEN') or os.environ.get('GH_TOKEN')
GH_HEADERS = {'Authorization': f'Bearer {_gh_token}'} if _gh_token else {}

conn = connect()
cursor = conn.cursor()

def query(sql):
    cursor.execute(sql)
    cols = [c[0] for c in cursor.description]
    return pd.DataFrame(cursor.fetchall(), columns=cols)

print(f"Report generated: {NOW.isoformat(timespec='seconds')}")
print(f"Today (UTC): {TODAY} ({TODAY.strftime('%A')})")

In [ ]:
# One entry per gold table: where it lives at each layer, its date column,
# and how often it's genuinely expected to move. 'trading_day' sources only
# advance on US market days - this check approximates that with a Mon-Fri
# rule and does not know about market holidays, so a holiday will show as
# one extra 'stale' day and should be read as expected, not a real gap.
TABLE_REGISTRY = {
    'market_daily':     {'bronze': 'workspace.bronze.market_data',    'silver': 'workspace.silver.market_data',    'gold': 'workspace.gold.market_daily',     'date_col': 'date',          'cadence': 'trading_day'},
    'sector_rotation':  {'bronze': 'workspace.bronze.market_data',    'silver': 'workspace.silver.market_data',    'gold': 'workspace.gold.sector_rotation',  'date_col': 'date',          'cadence': 'trading_day'},
    'ai_vs_market':     {'bronze': 'workspace.bronze.market_data',    'silver': 'workspace.silver.market_data',    'gold': 'workspace.gold.ai_vs_market',     'date_col': 'date',          'cadence': 'trading_day'},
    'ai_basket_detail': {'bronze': 'workspace.bronze.market_data',    'silver': 'workspace.silver.market_data',    'gold': 'workspace.gold.ai_basket_detail', 'date_col': 'date',          'cadence': 'trading_day'},
    'volatility':       {'bronze': 'workspace.bronze.market_data',    'silver': 'workspace.silver.market_data',    'gold': 'workspace.gold.volatility',       'date_col': 'date',          'cadence': 'trading_day'},
    'macro_snapshot':   {'bronze': 'workspace.bronze.macro_data',     'silver': 'workspace.silver.macro_data',     'gold': 'workspace.gold.macro_snapshot',   'date_col': 'date',          'cadence': 'mixed'},
    'attention_index':  {'bronze': 'workspace.bronze.attention_data', 'silver': 'workspace.silver.attention_data', 'gold': 'workspace.gold.attention_index',  'date_col': 'date',          'cadence': 'daily_lagged'},
    'dev_momentum':     {'bronze': 'workspace.bronze.dev_momentum',   'silver': 'workspace.silver.dev_momentum',   'gold': 'workspace.gold.dev_momentum',     'date_col': 'snapshot_date', 'cadence': 'daily'},
    'research_pace':    {'bronze': 'workspace.bronze.research_pace',  'silver': 'workspace.silver.research_pace',  'gold': 'workspace.gold.research_pace',    'date_col': 'snapshot_date', 'cadence': 'daily'},
}

## 1. Did the pipeline actually run?

Pulled directly from the GitHub Actions API rather than inferred from the data, since a run can report success while still landing incomplete data (this happened for real on 2026-08-03 - see `PROJECT_MEMORY.md` bug #19). This section checks the run history independently of the data checks in Section 2.

In [ ]:
runs_resp = requests.get(
    f'https://api.github.com/repos/{REPO}/actions/workflows/pipeline.yml/runs',
    params={'per_page': 30}, headers=GH_HEADERS, timeout=30,
)
runs_resp.raise_for_status()
runs = runs_resp.json()['workflow_runs']

run_rows = []
for r in runs:
    run_rows.append({
        'date': r['created_at'][:10],
        'time_utc': r['created_at'][11:16],
        'event': r['event'],
        'status': r['conclusion'] or r['status'],
        'url': r['html_url'],
    })
runs_df = pd.DataFrame(run_rows)

# one row per calendar day: the last run that day (a manual re-run after a
# failure should be what counts, not the earlier failed attempt)
runs_by_day = runs_df.sort_values('time_utc').groupby('date').last().sort_index(ascending=False)
last_7_days = [(TODAY - timedelta(days=i)).isoformat() for i in range(7)]
recent_runs = runs_by_day.reindex(last_7_days)
print('Last 7 calendar days (most recent run per day, if any):')
recent_runs[['time_utc', 'event', 'status']]

In [ ]:
today_str = TODAY.isoformat()
today_status = runs_by_day['status'].get(today_str)

issues_resp = requests.get(
    f'https://api.github.com/repos/{REPO}/issues',
    params={'labels': 'pipeline-failure', 'state': 'open'}, headers=GH_HEADERS, timeout=30,
)
issues_resp.raise_for_status()
open_failure_issues = issues_resp.json()

if today_status == 'success':
    run_line = f'Today\'s pipeline run succeeded.'
elif today_status is None:
    run_line = 'No pipeline run recorded yet today.'
else:
    run_line = f'Today\'s most recent run reported **{today_status}**.'

if open_failure_issues:
    issue_line = (
        f"There {'is' if len(open_failure_issues) == 1 else 'are'} {len(open_failure_issues)} open "
        f"pipeline-failure issue(s), most recently [{open_failure_issues[0]['title']}]"
        f"({open_failure_issues[0]['html_url']})."
    )
else:
    issue_line = 'No open pipeline-failure issues.'

display(Markdown(f'**Run status:** {run_line} {issue_line}'))

## 2. What data landed, and does it look right?

Queried directly from the gold tables in Databricks, not the published JSON - this checks the database itself, which is the source of truth the JSON export is only a copy of.

In [ ]:
def most_recent_weekday(d):
    while d.weekday() >= 5:  # Saturday=5, Sunday=6
        d -= timedelta(days=1)
    return d

def expected_trading_date(now_utc):
    # Before ~21:00 UTC, today's own session may not have closed yet, so
    # the most recent day we can reasonably expect data for is the prior
    # weekday, not today - this is what makes a Sunday-morning run with
    # Friday's close correctly read as FRESH rather than 2 days stale.
    d = now_utc.date()
    if now_utc.hour < 21:
        d -= timedelta(days=1)
    return most_recent_weekday(d)

def expected_status(cadence, latest_date):
    if latest_date is None:
        return 'NO DATA'
    gap_days = (TODAY - latest_date).days
    if cadence == 'trading_day':
        expected = expected_trading_date(NOW)
        return 'FRESH' if latest_date >= expected else 'STALE'
    if cadence == 'daily_lagged':
        # attention_index: Wikimedia typically lags 1-2 days, see bug #18
        return 'FRESH' if gap_days <= 2 else 'STALE'
    if cadence == 'daily':
        return 'FRESH' if gap_days <= 1 else 'STALE'
    if cadence == 'mixed':
        # macro_snapshot mixes monthly (CPI, unemployment, fed funds) and
        # near-daily (10Y yield) series - checked per-series below instead
        # of with a single blanket rule.
        return 'SEE PER-SERIES'
    return 'UNKNOWN'

summary_rows = []
for name, spec in TABLE_REGISTRY.items():
    df = query(f"SELECT MAX({spec['date_col']}) AS latest, COUNT(*) AS rows FROM {spec['gold']}")
    latest = df['latest'].iloc[0]
    latest_date = latest.date() if hasattr(latest, 'date') else latest
    summary_rows.append({
        'table': name,
        'gold': spec['gold'],
        'latest_date': latest_date,
        'total_rows': int(df['rows'].iloc[0]),
        'cadence': spec['cadence'],
        'status': expected_status(spec['cadence'], latest_date),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df

In [ ]:
# macro_snapshot checked per-series, since CPI/unemployment/fed funds are
# monthly and 10Y yield is near-daily - one blanket freshness rule would
# either wrongly flag the monthly series every day or hide a real gap in
# the daily one.
macro_df = query("SELECT series, MAX(date) AS latest, COUNT(*) AS rows FROM workspace.gold.macro_snapshot GROUP BY series ORDER BY series")
MONTHLY_SERIES = {'cpi', 'unemployment_rate', 'fed_funds_rate'}

def macro_status(series, latest_date):
    gap_days = (TODAY - latest_date).days
    if series in MONTHLY_SERIES:
        return 'FRESH' if gap_days <= 45 else 'STALE'
    return 'FRESH' if gap_days <= 5 else 'STALE'

macro_df['latest_date'] = macro_df['latest'].apply(lambda v: v.date() if hasattr(v, 'date') else v)
macro_df['status'] = macro_df.apply(lambda r: macro_status(r['series'], r['latest_date']), axis=1)
macro_df[['series', 'latest_date', 'rows', 'status']]

In [ ]:
stale_tables = summary_df[summary_df['status'] == 'STALE']['table'].tolist()
stale_macro = macro_df[macro_df['status'] == 'STALE']['series'].tolist()
all_stale = stale_tables + [f'macro:{s}' for s in stale_macro]

if all_stale:
    display(Markdown(f'**Data freshness:** {len(all_stale)} source(s) look stale for today\'s cadence: {", ".join(all_stale)}. Worth checking the run log for the actual failure.'))
else:
    display(Markdown('**Data freshness:** every source is as fresh as expected for its own cadence today.'))

## 3. What today's numbers mean

A plain-language read of the latest reading from each source, generated from whatever is actually in the tables right now.

In [ ]:
lines = []

market = query("SELECT symbol, date, close, daily_return FROM workspace.gold.market_daily WHERE symbol = '^GSPC' ORDER BY date DESC LIMIT 1")
if not market.empty:
    r = market.iloc[0]
    direction = 'up' if (r['daily_return'] or 0) >= 0 else 'down'
    pct = abs(r['daily_return'] or 0) * 100
    lines.append(f"- **Market**: S&P 500 closed at {r['close']:,.2f} on {r['date']}, {direction} {pct:.2f}% from the prior session.")

avm = query('SELECT date, ai_basket_return, benchmark_return, spread FROM workspace.gold.ai_vs_market ORDER BY date DESC LIMIT 1')
if not avm.empty and avm.iloc[0]['spread'] is not None:
    r = avm.iloc[0]
    lead = 'outperformed' if r['spread'] >= 0 else 'underperformed'
    lines.append(f"- **AI basket**: {lead} the S&P 500 by {abs(r['spread'])*100:.2f} points on {r['date']} ({r['ai_basket_return']*100:+.2f}% vs. {r['benchmark_return']*100:+.2f}%).")

vol = query('SELECT date, rolling_20d_volatility FROM workspace.gold.volatility WHERE rolling_20d_volatility IS NOT NULL ORDER BY date DESC LIMIT 1')
if not vol.empty:
    r = vol.iloc[0]
    lines.append(f"- **Volatility**: rolling 20-day realized volatility is {r['rolling_20d_volatility']*100:.2f}% as of {r['date']}.")

for series_name, label in [('cpi', 'CPI'), ('unemployment_rate', 'Unemployment'), ('fed_funds_rate', 'Fed funds rate'), ('10y_yield', '10Y yield')]:
    row = macro_df[macro_df['series'] == series_name]
    if not row.empty:
        latest_row = query(f"SELECT date, value FROM workspace.gold.macro_snapshot WHERE series = '{series_name}' ORDER BY date DESC LIMIT 1")
        if not latest_row.empty:
            r = latest_row.iloc[0]
            lines.append(f"- **{label}**: latest reading is {r['value']:,.2f} as of {r['date']}.")

attn = query('SELECT article, date, views FROM workspace.gold.attention_index ORDER BY date DESC LIMIT 3')
if not attn.empty:
    parts = ', '.join(f"{r['article'].replace('_', ' ')}: {int(r['views']):,}" for _, r in attn.iterrows())
    lines.append(f"- **Public attention**: most recent pageviews ({attn.iloc[0]['date']}) - {parts}.")

dev = query('SELECT repo, snapshot_date, stars FROM workspace.gold.dev_momentum WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM workspace.gold.dev_momentum) ORDER BY stars DESC')
if not dev.empty:
    top = dev.iloc[0]
    lines.append(f"- **Dev momentum**: as of {top['snapshot_date']}, {top['repo']} leads the tracked repos at {int(top['stars']):,} stars.")

research = query('SELECT category, snapshot_date, count FROM workspace.gold.research_pace ORDER BY snapshot_date DESC LIMIT 2')
if not research.empty:
    parts = ', '.join(f"{r['category']}: {int(r['count'])}" for _, r in research.iterrows())
    lines.append(f"- **Research pace**: trailing-7d arXiv counts as of {research.iloc[0]['snapshot_date']} - {parts}.")

display(Markdown('\n'.join(lines)))

## 4. Where everything lives

Static reference - these table names and the pipeline stages between them don't change day to day, unlike the freshness checks above.

| Gold table | Silver source | Bronze source | Published as | Dashboard panel |
|---|---|---|---|---|
| `workspace.gold.market_daily` | `workspace.silver.market_data` | `workspace.bronze.market_data` | `market_daily.json` | Market Snapshot |
| `workspace.gold.sector_rotation` | `workspace.silver.market_data` | `workspace.bronze.market_data` | `sector_rotation.json` | Sector Rotation |
| `workspace.gold.volatility` | `workspace.silver.market_data` | `workspace.bronze.market_data` | `volatility.json` | Volatility |
| `workspace.gold.macro_snapshot` | `workspace.silver.macro_data` | `workspace.bronze.macro_data` | `macro_snapshot.json` | Macro Backdrop |
| `workspace.gold.ai_vs_market` | `workspace.silver.market_data` | `workspace.bronze.market_data` | `ai_vs_market.json` | AI Pulse (spread chart) |
| `workspace.gold.ai_basket_detail` | `workspace.silver.market_data` | `workspace.bronze.market_data` | `ai_basket_detail.json` | AI Pulse (composition donut) |
| `workspace.gold.attention_index` | `workspace.silver.attention_data` | `workspace.bronze.attention_data` | `attention_index.json` | AI Pulse (attention trend) |
| `workspace.gold.dev_momentum` | `workspace.silver.dev_momentum` | `workspace.bronze.dev_momentum` | `dev_momentum.json` | AI Pulse (dev momentum) |
| `workspace.gold.research_pace` | `workspace.silver.research_pace` | `workspace.bronze.research_pace` | `research_pace.json` | AI Pulse (research pace) |

Bronze is overwritten in full every run (a snapshot of the latest raw pull, not accumulated history). Silver accumulates via `MERGE` keyed by natural key (`symbol`+`date`, `series`+`date`, etc.), so it holds the full history. Gold is fully recomputed from silver on every run. Full detail on this design is in `PROJECT_PLAN.md` Section 6 and `PROJECT_MEMORY.md`.

## Overall health

In [ ]:
run_ok = today_status == 'success' and not open_failure_issues
data_ok = not all_stale

if run_ok and data_ok:
    verdict = 'HEALTHY - today\'s run succeeded and every source is as fresh as expected.'
elif not run_ok and data_ok:
    verdict = 'CHECK RUN LOG - the data looks fine, but the run status or an open failure issue says otherwise. See Section 1.'
elif run_ok and not data_ok:
    verdict = 'CHECK DATA - the run reported success, but at least one source looks stale for its cadence (this is exactly how bug #19 slipped through). See Section 2.'
else:
    verdict = 'NEEDS ATTENTION - both the run status and the data freshness checks are flagging a problem.'

display(Markdown(f'**{verdict}**'))

cursor.close()
conn.close()